In [ ]:
!pip install great_tables

# Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import warnings
import scipy
from sklearn.compose import TransformedTargetRegressor
from sklearn import set_config
from colorama import Style, Fore
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.model_selection import StratifiedKFold, KFold
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge, LinearRegression
from lightgbm import LGBMRegressor
from category_encoders import TargetEncoder, OneHotEncoder, MEstimateEncoder, OrdinalEncoder
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import roc_auc_score, roc_curve, make_scorer, mean_squared_log_error, r2_score
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone
from sklearn.preprocessing import FunctionTransformer, StandardScaler, LabelEncoder, LabelBinarizer, MinMaxScaler, PolynomialFeatures, SplineTransformer
from sklearn.compose import ColumnTransformer
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from catboost import CatBoostRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, VotingRegressor, RandomForestRegressor

from great_tables import GT, style ,exibble, from_column, loc
from colorama import Style, Fore

sns.set_theme(style = 'white', palette = 'colorblind')
pal = sns.color_palette('colorblind')

pd.set_option('display.max_rows', 100)
set_config(transform_output = 'pandas')
pd.options.mode.chained_assignment = None
warnings.simplefilter(action='ignore', category=FutureWarning)

# Config

In [ ]:
palette = ['#20B2AA', '#D8BFD8', '#8B0000', '#C09741',
           '#EC5B6D', '#90A6B1', '#6ca957', '#D8E3E2']

config = {
    'SEED' : 42,
    'N_SPLITS': 5,
    'SUBMIT' : True,
    'USE_ORIGINAL': False
    
}

# Functions


In [ ]:
def printColor(pText: str):
    print(f'{Style.BRIGHT}{Fore.GREEN}{pText}{Style.RESET_ALL}')    

In [ ]:
def printInfo():
    print(f'{Style.BRIGHT}{Fore.YELLOW}SHAPE{Style.RESET_ALL}')
    print(f'{Style.BRIGHT}{Fore.GREEN} train: {train.shape}')
    print(f'{Style.BRIGHT}{Fore.GREEN} test:  {test.shape}')
    print(f'{Style.BRIGHT}{Fore.GREEN} original:  {original.shape}')
    print(f'{Style.BRIGHT}{Fore.YELLOW}\nNULL VALUES{Style.RESET_ALL}')
    print(f'{Style.BRIGHT}{Fore.GREEN} train: {train.isnull().any().any()}')
    print(f'{Style.BRIGHT}{Fore.GREEN} train: {test.isnull().any().any()}')
    print(f'{Style.BRIGHT}{Fore.GREEN} original: {original.isnull().any().any()}')    
    print(f'{Style.BRIGHT}{Fore.YELLOW}\nDUPLICATES{Style.RESET_ALL}')
    print(f'{Style.BRIGHT}{Fore.GREEN} train: {train.duplicated().any().any()}')
    print(f'{Style.BRIGHT}{Fore.GREEN} train: {test.duplicated().any().any()}')
    print(f'{Style.BRIGHT}{Fore.GREEN} original: {original.duplicated().any().any()}')    

In [ ]:
def customStatistic(df: pd.DataFrame(), categoric = False):
    num_cols = list(df._get_numeric_data())
    cat_cols = list(df.drop(num_cols,axis=1))
    if categoric:
        desc = pd.DataFrame(index = list(df[cat_cols]))
        df = df[cat_cols]
    else:
        desc = pd.DataFrame(index = list(df[num_cols]))
        df = df[num_cols]
        desc['skew'] = df[num_cols].skew()
        
    desc['type'] = df.dtypes
    desc['count'] = df.count()
    desc['nunique'] = df.nunique()
    desc['%unique'] = desc['nunique'] /len(df) * 100 
    desc['null'] = df.isnull().sum()
    desc['%null'] = desc['null'] / len(df) * 100
    desc = pd.concat([desc,df.describe().T.drop('count',axis=1)],axis=1)    

    desc = desc.round(2)
    return desc.reset_index().rename(columns={'index':'Column'}).sort_values(by=['type'])

In [ ]:
def min_max_unique(data_train, data_test):
    
    df = pd.DataFrame(index=data_train.columns)
    summary = {}
    for col in data_train.columns:
        if pd.api.types.is_numeric_dtype(data_train[col]):  # Verifica se a coluna é numérica
            min_train = min(data_train[col])
            min_test = min(data_test[col])
            max_train = max(data_train[col])
            max_test = max(data_test[col])
            unique_train = len(data_train[col].unique())
            unique_test = len(data_test[col].unique())
            top5_train = sorted(data_train[col])[:5]
            top5_test = sorted(data_test[col])[:5]
        else:  
            min_train = min_test = max_train = max_test = None
            unique_train = len(data_train[col].unique())
            unique_test = len(data_test[col].unique())
            top5_train = top5_test = None
        summary[col] = [min_train, min_test, max_train, max_test, 
                        unique_train, unique_test]

    df = pd.DataFrame.from_dict(summary, orient='index', columns=['min_train', 'min_test', 'max_train', 'max_test', 
                                                                  'unique_train', 'unique_test'])\
        .reset_index().rename(columns={'index': 'columns'})


    return df

# Data

In [ ]:
train = pd.read_csv(r'/kaggle/input/playground-series-s4e5/train.csv', index_col = 'id')
test = pd.read_csv(r'/kaggle/input/playground-series-s4e5/test.csv', index_col = 'id')
sub = pd.read_csv(r'/kaggle/input/playground-series-s4e5/sample_submission.csv', index_col = 'id')
original = pd.read_csv(r'/kaggle/input/flood-prediction-factors/flood.csv')

In [ ]:
train.head(3)

In [ ]:
test.head(3)

In [ ]:
printInfo()

In [ ]:
TARGET = 'FloodProbability'
NUMERIC_COLS = [f for f in train._get_numeric_data() if f not in TARGET]
CAT_COLS = list(test.drop(NUMERIC_COLS,axis=1))
print(f'Numeric cols: {len(NUMERIC_COLS)}')
print(f'Cat cols: {len(CAT_COLS)}')

# Descriptive Statistics

In [ ]:
stat = customStatistic(train,False)
GT(stat)\
    .tab_header(title='Descriptive Statistic - Train', subtitle='Numeric Fields')\
    .data_color(columns=['min','max','mean'],palette=['lightblue','lightcoral'],alpha=0.5)\
    .fmt_percent(columns=['%unique','%null'])

In [ ]:
stat = customStatistic(test,False)
GT(stat)\
    .tab_header(title='Descriptive Statistic - Test', subtitle='Numeric Fields')\
    .data_color(columns=['min','max','mean'],palette=['lightblue','lightcoral'],alpha=0.5)\
    .fmt_percent(columns=['%unique','%null'])

In [ ]:
s = min_max_unique(train.drop(TARGET,axis=1),test)
GT(s)\
    .tab_header(title='Min Max Uniques', subtitle='Train and Test datasets')\
    .data_color(columns=['columns'],palette=['lightgray','lightgray'])
    

# Correlation

In [ ]:
def plot_correlation(df,label=''):
    corr = df._get_numeric_data().corr(method='spearman')
    mask = np.zeros_like(corr)
    mask[np.triu_indices_from(mask)]=True
    fig, ax = plt.subplots(figsize=(25,17))
    sns.heatmap(data=corr, 
                mask=mask , 
                annot=True,
                cmap='icefire',
                annot_kws={'size': 12, 'rotation': 45},
                ax=ax
                );
    ax.set_title(f'Correlation {label}',fontsize=25, fontweight='bold');

In [ ]:
plot_correlation(train,'train')

<div style="border-radius: 10px; border:#27374D solid; padding: 15px; background-color: #ffffff00; font-size: 100%; text-align: left;">
All features are correlated with the target.
</div>

# EDA (Numeric Fields)

In [ ]:
def plot_numerical():
    #num = train.select_dtypes(include=['int64','float64']).columns

    df = pd.concat([train[NUMERIC_COLS].assign(Source = 'Train'), 
                    test[NUMERIC_COLS].assign(Source = 'Test')], ignore_index = True)

    # Use of more advanced artistic matplotlib interface (see the axes)
    fig, axes = plt.subplots(len(NUMERIC_COLS), 3 ,figsize = (16, len(NUMERIC_COLS) * 4), 
                             gridspec_kw = {'hspace': 0.35, 'wspace': 0.3, 
                                            'width_ratios': [0.80, 0.20, 0.20]})

    for i,col in enumerate(NUMERIC_COLS):
        ax = axes[i,0]
        sns.kdeplot(data = df[[col, 'Source']], x = col, hue = 'Source', palette=['#456cf0', '#ed7647'], linewidth = 2.1, warn_singular=False, ax = ax) # Use of seaborn with artistic interface
        ax.set_title(f"\n{col}",fontsize = 9)
        ax.grid(visible=True, which = 'both', linestyle = '--', color='lightgrey', linewidth = 0.75)
        ax.set(xlabel = '', ylabel = '')

        ax = axes[i,1]
        sns.boxplot(data = df.loc[df.Source == 'Train', [col]], y = col, width = 0.25, linewidth = 0.90, fliersize= 2.25, color = '#456cf0', ax = ax)
        ax.set(xlabel = '', ylabel = '')
        ax.set_title("Train", fontsize = 9)

        ax = axes[i,2]
        sns.boxplot(data = df.loc[df.Source == 'Test', [col]], y = col, width = 0.25, linewidth = 0.90, fliersize= 2.25, color = '#ed7647', ax = ax)
        ax.set(xlabel = '', ylabel = '')
        ax.set_title("Test", fontsize = 9)

    plt.suptitle(f'\nDistribution analysis - numerical features',fontsize = 12, y = 0.89, x = 0.57, fontweight='bold')
    plt.show()

In [ ]:
plot_numerical()

<div style="border-radius: 10px; border:#27374D solid; padding: 15px; background-color: #ffffff00; font-size: 100%; text-align: left;">
training and testing have similar distributions
</div>

# EDA (Target)

In [ ]:
sns.kdeplot(train[TARGET], fill=True);
plt.title('Target distribuition');

In [ ]:
train[TARGET].skew()

<div style="border-radius: 10px; border:#27374D solid; padding: 15px; background-color: #ffffff00; font-size: 100%; text-align: left;">
approaches a normal distribution
</div>

# Cross validation

In [ ]:
scores, oof, test_preds = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

In [ ]:
kf = KFold(n_splits=config['N_SPLITS'], random_state=config['SEED'],shuffle=True)

In [ ]:
def score_model(estimator, label = ''):
    
    X = train.copy()
    y = X.pop(TARGET)
    
    val_predictions = np.zeros((len(X)))
    test_predictions = np.zeros((len(test)))
    val_scores= []  
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        
        model = clone(estimator)

        X_train = X.iloc[train_idx].reset_index(drop = True)
        y_train = y.iloc[train_idx].reset_index(drop = True)

        X_val = X.iloc[val_idx].reset_index(drop = True)
        y_val = y.iloc[val_idx].reset_index(drop = True)
        if config['USE_ORIGINAL']:
            X_train = pd.concat([X_train, original.drop([TARGET],axis=1)])
            y_train = pd.concat([y_train,original[TARGET]])
            
        model.fit(X_train, y_train)      
        val_preds = model.predict(X_val)
        val_predictions[val_idx] += val_preds
        test_predictions += model.predict(test) / kf.get_n_splits()
                
        val_score = r2_score(y_val, val_preds)
        val_scores.append(val_score)

        print(f'Fold {fold+1}: {val_score:.5f}')
    
    printColor(f'Val Score: {np.mean(val_scores):.5f} ± {np.std(val_scores):.5f} | {label}')
         

    return val_scores, val_predictions, test_predictions

# Models

<div style="border-radius: 10px; border:#27374D solid; padding: 15px; background-color: #ffffff00; font-size: 100%; text-align: left;">
Using new fetures related in <a href="https://www.kaggle.com/code/carlmcbrideellis/automl-grand-prix-cme-catboost-baseline-5">Notebook</a> 
</div>

In [ ]:
class FE(BaseEstimator, TransformerMixin):
    
    def __init__(self):
        pass
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        x_copy = X.copy()
        features = x_copy.columns.tolist()
        x_copy['mean_features'] = 0.1*x_copy[features].mean(axis=1)
        x_copy['std_features'] = x_copy[features].std(axis=1)
        x_copy['max_features'] = x_copy[features].max(axis=1)
        x_copy['min_features'] = x_copy[features].min(axis=1)
        x_copy['median_features'] = 0.1*x_copy[features].median(axis=1)
        x_copy['sum_features'] = X[NUMERIC_COLS].sum(axis=1)
        #x_copy['special1'] = x_copy['sum_features'].isin(np.arange(72, 76))        
        #x_copy['ptp'] = x_copy[NUMERIC_COLS].values.ptp(axis=1)
        sorted_features = [f'sort_{i}' for i in np.arange(len(NUMERIC_COLS))]
        x_copy[sorted_features] = np.sort(x_copy[NUMERIC_COLS],axis=1)
        x_copy['q1'] = x_copy[features].quantile(0.25,axis=1)
        x_copy['q2'] = x_copy[features].quantile(0.50,axis=1)
        x_copy['q3'] = x_copy[features].quantile(0.75,axis=1)
        
        
        x_copy = x_copy.drop(features, axis=1)      
                                                            
        return x_copy                                                                

In [ ]:
%%time
scores['XGB'],oof['XGB'],test_preds['XGB'] = score_model(make_pipeline(
                                                            FE(),
                                                            XGBRegressor(n_estimators=1000,
                                                                         max_depth=5,
                                                                         learning_rate=0.04,
                                                                         random_state=config['SEED'])),
                                                        'XGB')                                    


In [ ]:
# scores['LinearRegression'],oof['LinearRegression'],test_preds['LinearRegression'] = score_model(
#                                                                        make_pipeline(FE(),
#                                                                                      StandardScaler(),
#                                                                                      LinearRegression()),
#                                                                        'LinearRegression')                                    

In [ ]:
# scores['Ridge'],oof['Ridge'],test_preds['Ridge'] = score_model(make_pipeline(   
#                                                                FE(), 
#                                                                StandardScaler(),
#                                                                PolynomialFeatures(2), 
#                                                                LinearRegression()),
#                                                                'Ridge')                                    

In [ ]:
%%time
scores['CatBoost'],oof['CatBoost'],test_preds['CatBoost'] = score_model(make_pipeline(
                                                                        FE(),                                                                              
                                                                        CatBoostRegressor(iterations=2000,
                                                                                          verbose=False, 
                                                                                          random_state=config['SEED'])),
                                                                       'CatBoost')                                    

In [ ]:
%%time
lgbm_params = {
    'num_leaves': 183, 
    'learning_rate': 0.01183688880802108, 
    'n_estimators': 600, 
    'subsample_for_bin': 165697, 
    'min_child_samples': 114, 
    'reg_alpha': 2.075080888948164e-06, 
    'reg_lambda': 3.838938366471552e-07, 
    'colsample_bytree': 0.9634044234652241, 
    'subsample': 0.9592138618622019, 
    'max_depth': 9,
    'random_state':config['SEED'],
    'verbose': -1
}
scores['LGBM'],oof['LGBM'],test_preds['LGBM'] = score_model(
                                                         make_pipeline(
                                                          FE(),
                                                          LGBMRegressor(**lgbm_params)),
                                                         'LGBM')                                    


# Ensemble

In [ ]:
def objective_weights(trial):
    multipliers = np.array([trial.suggest_float(f'{label}', -1, 1) for label in list(oof)])
    
    return r2_score(train[TARGET], oof.to_numpy() @ (multipliers / multipliers.sum()))

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(sampler=optuna.samplers.TPESampler(seed=config['SEED']),
                   study_name='weights',direction ='maximize')
study.optimize(objective_weights,n_trials=1000)

In [ ]:
print(f'R2 -> {study.best_value}')
print(f'{study.best_params}')

In [ ]:
w = np.asarray([ p[1] for p in study.best_params.items()])
w /= w.sum()
w

In [ ]:
scores['Ensemble'] = r2_score(train[TARGET],oof.to_numpy() @ w)

# Scores

In [ ]:
ax = scores.mean().sort_values(ascending=True).plot(kind='barh', figsize=(10, 6), color='#a2a28f')

for container in ax.containers:
    ax.bar_label(container, label_type='center', color='black', fontsize=12, fontweight='bold')
ax.patches[-1].set_facecolor('#6ca957')

ax.set_title('Score Models', fontsize=16, fontweight='bold')

ax.set_xlabel('R2', fontsize=14, fontweight='bold')
ax.set_ylabel('Models', fontsize=14, fontweight='bold')

ax.tick_params(axis='both', which='major', labelsize=12)


ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()

# Submission

In [ ]:
predictions = test_preds.to_numpy() @ w

In [ ]:
submission = test.copy()
submission[TARGET] = predictions
submission[TARGET].hist()
if config['SUBMIT']:
    submission[TARGET].to_csv('submission.csv')